# 🟫 Capa Bronze: Ingesta Perimetral de Datos Crudos
Este notebook automatiza la extracción de datos desde las APIs de ENTSO-E (España y Rumanía), SMARD (Alemania) y PSE (Polonia), persistiendo los payloads originales sin ninguna transformación para garantizar la trazabilidad completa del dato.

In [1]:
import requests
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta

# Configuración por país — añadir nuevos mercados aquí sin tocar el resto del pipeline
CONFIG_PAISES = {
    "espana": {
        "url": "https://web-api.tp.entsoe.eu/api",
        "params": {
            "documentType": "A44",
            "in_Domain": "10YES-REE------0",
            "out_Domain": "10YES-REE------0",
            "securityToken": "3cd4e295-9856-40a6-9d4a-ce3844396ac9"
        }
    },
    "rumania": {
        "url": "https://web-api.tp.entsoe.eu/api",
        "params": {
            "documentType": "A44",
            "in_Domain": "10YRO-TEL------P",
            "out_Domain": "10YRO-TEL------P",
            "securityToken": "3cd4e295-9856-40a6-9d4a-ce3844396ac9"
        }
    },
    "alemania": {
        "url_indice": "https://www.smard.de/app/chart_data/4169/DE/index_hour.json",
        "url_bloque": "https://www.smard.de/app/chart_data/4169/DE/4169_DE_hour_{timestamp}.json"
    },
    "polonia": {
        "url": "https://api.raporty.pse.pl/api/rce-pln"
    }
}

# Ventana de 6 días para cubrir retrasos de publicación (ENTSO-E D-1, SMARD hasta D-2)
fecha_fin = datetime.utcnow()
fecha_inicio = fecha_fin - timedelta(days=6)
print(f"Intervalo de extracción incremental: {fecha_inicio.strftime('%Y-%m-%d')} al {fecha_fin.strftime('%Y-%m-%d')}")

StatementMeta(, eae2b1f9-b153-4653-a688-be009051d9fb, 3, Finished, Available, Finished, False)

Intervalo de extracción incremental: 2026-06-06 al 2026-06-12


In [2]:
def descargar_entsoe(config, f_inicio, f_fin):
    params = config["params"].copy()
    params["periodStart"] = f_inicio.strftime("%Y%m%d%H00")
    params["periodEnd"] = f_fin.strftime("%Y%m%d%H00")
    response = requests.get(config["url"], params=params)
    if response.status_code == 200:
        return response.text
    raise Exception(f"Error en ENTSO-E: {response.status_code}")

def descargar_smard(config):
    res_idx = requests.get(config["url_indice"])
    if res_idx.status_code != 200: 
        raise Exception("Error al consultar el índice de SMARD")
    timestamps = res_idx.json().get("timestamps", [])[:2]  # Últimos 2 bloques semanales (~14 días de cobertura)
    bloques = []
    for ts in timestamps:
        res_blq = requests.get(config["url_bloque"].format(timestamp=ts))
        if res_blq.status_code == 200: 
            bloques.append(res_blq.json())
    return bloques

def descargar_polonia(config, f_inicio, f_fin):
    datos = []
    delta = f_fin - f_inicio
    for i in range(delta.days + 1):
        dia = (f_inicio + timedelta(days=i)).strftime("%Y-%m-%d")
        res = requests.get(f"{config['url']}?$filter=business_date eq '{dia}'")
        if res.status_code == 200: 
            datos.extend(res.json().get("value", []))
    return datos

StatementMeta(, eae2b1f9-b153-4653-a688-be009051d9fb, 4, Finished, Available, Finished, False)

In [3]:
print("📥 Iniciando descarga perimetral de mercados...")

# España y Rumanía comparten endpoint ENTSO-E, se procesan en bucle
for pais in ["espana", "rumania"]:
    xml_data = descargar_entsoe(CONFIG_PAISES[pais], fecha_inicio, fecha_fin)
    spark.createDataFrame([(xml_data,)], ["raw_xml"]).write.mode("overwrite").saveAsTable(f"bronze_raw_{pais}")

# Alemania — bloques semanales SMARD deserializados desde lista de dicts
json_smard = descargar_smard(CONFIG_PAISES["alemania"])
spark.read.json(spark.sparkContext.parallelize([str(json_smard)])).write.mode("overwrite").saveAsTable("bronze_raw_alemania")

# Polonia — guardado condicional por si la API no devuelve registros
json_polonia = descargar_polonia(CONFIG_PAISES["polonia"], fecha_inicio, fecha_fin)
if json_polonia:
    spark.createDataFrame(json_polonia).write.mode("overwrite").saveAsTable("bronze_raw_polonia")

print("✅ Capa Bronze consolidada e inmutable.")

StatementMeta(, eae2b1f9-b153-4653-a688-be009051d9fb, 5, Finished, Available, Finished, False)

📥 Iniciando descarga perimetral de mercados...
✅ Capa Bronze consolidada e inmutable.
